In [ ]:
# @title 1. Build Edge Environment (Run this first)
import os

print("➔ 1. Cloning Repository...")
!git clone https://github.com/DhakshanaS/Thermal-Edge-AI.git
%cd Thermal-Edge-AI

print("➔ 2. Installing C++ TensorRT Dependencies...")
!wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb
!dpkg -i cuda-keyring_1.1-1_all.deb > /dev/null
!apt-get update > /dev/null
!DEBIAN_FRONTEND=noninteractive apt-get install -yq tzdata libnvinfer-dev tensorrt > /dev/null

print("➔ 3. Building ONNX into a TensorRT Engine...")
!TRTEXEC=$(find /usr -name trtexec -type f | grep -v "site-packages" | head -n 1) && $TRTEXEC --onnx=models/best.onnx --saveEngine=models/thermal_cpp.engine > /dev/null

print("➔ 4. Validating Deployment Path via CMake...")
os.makedirs('build', exist_ok=True)
%cd build
!cmake ../src > /dev/null
!make > /dev/null
print("\n➔ 5. Running C++ Memory Validation:")
!./thermal_engine
%cd ..

print("\n✅ Edge Environment Successfully Built!")

In [ ]:
# @title 2. Launch Live Real-Time Demo
print("➔ Installing Python dependencies...")
!pip install Flask pyngrok ultralytics opencv-python-headless tensorrt > /dev/null
from pyngrok import ngrok

print("Get a free ngrok token at https://dashboard.ngrok.com/get-started/your-authtoken")
NGROK_TOKEN = input("Paste your ngrok Auth Token here: ")
!ngrok config add-authtoken {NGROK_TOKEN} > /dev/null

ngrok.kill()
public_url = ngrok.connect(5000)

print("\n" + "="*70)
print(f"🚀 LIVE C++ EDGE DEMO READY!")
print(f"🔗 CLICK HERE: {public_url}")
print("="*70 + "\n")

%cd /content/Thermal-Edge-AI
!python web/app.py